# Scraping via DTS - Thunderdots

> **This course presents the possibilities offered by the DTS API and the ThunderDots Python client for DTS endpoints for web scraping and text processing.**

### Distributed Text Services with ThunderDots

[Distributed Text Services](https://dtsapi.org/specifications/) (DTS) is a standard API for publishing and consuming collections of digital texts as machine-actionable data.

Rather than defining one API for a particular website, DTS defines a shared way of interacting with many textual repositories. A compatible client can therefore explore different DTS services without having to learn a completely different data model for each corpus.

DTS defines three main endpoints:

- **Collection**: navigate through collections and discover textual resources;
- **Navigation**: inspect the internal citation structure of a text, such as books, chapters, sections or lines;
- **Document**: retrieve a complete text or a specific passage.

Collection and Navigation responses are generally represented as JSON-LD, while the Document endpoint commonly returns TEI/XML.

In this short example, we will inspect the collections made available by the École nationale des chartes and use [ThunderDots](https://dots-suite.github.io/ThunderDots/) to retrieve one of them.

#### ThunderDots

[ThunderDots](https://dots-suite.github.io/ThunderDots/) is a Python client for DTS endpoints.

It can:

- navigate collections and subcollections;
- retrieve DTS resources and their TEI/XML documents;
- extract complete texts or smaller fragments;
- preserve Dublin Core and extension metadata;
- export the resulting corpus to Python records, Pandas or Polars DataFrames;
- prepare records for indexing systems such as Elasticsearch or Qdrant.

Install it with:

```bash
pip install thunderdots pandas
```

The examples below use the DTS endpoint hosted by the École nationale des chartes:

```text
https://dots.chartes.psl.eu/api/dts
```

In [1]:
import sys

print(sys.executable)

/home/lconte/.pyenv/versions/3.12.1/envs/scraping/bin/python


In [2]:
import httpx
import pandas as pd

from IPython.display import JSON
from thunderdots import ThunderDots


DTS_ENDPOINT = "https://dev.chartes.psl.eu/dots/api/dts"

#### Discovering the available collections

A DTS service exposes an entry point describing the URLs of its Collection, Navigation and Document endpoints.

We first call the service entry point directly with `httpx`.

In [3]:
entrypoint_response = httpx.get(
    DTS_ENDPOINT,
    timeout=30,
    follow_redirects=True,
)

entrypoint_response.raise_for_status()

entrypoint_data = entrypoint_response.json()
JSON(entrypoint_data)

<IPython.core.display.JSON object>

The response normally contains links or URI templates for the DTS operations.

The exact URLs are implementation-dependent: DTS standardizes the operations and response models.

#### Inspecting the collections of the École nationale des chartes

The Collection endpoint is used to browse the hierarchy of collections and resources.

A collection response usually contains:

- `@id`: the collection identifier;
- `title`: its human-readable title;
- `@type`: generally `Collection` or `Resource`;
- `totalItems`: the number of direct members;
- `member`: the list of child collections or textual resources.

We call the root Collection route to discover what is currently available (we use a demo server here).

In [4]:
collection_endpoint=f"https://dev.chartes.psl.eu/dots/api/dts/collection"

collections_response = httpx.get(
    collection_endpoint,
    timeout=30,
    follow_redirects=True,
)

collections_response.raise_for_status()

collections_data = collections_response.json()
JSON(collections_data)

<IPython.core.display.JSON object>

In [5]:
# We can vizualize the JSON response into dataframe
collections = collections_data.get("member", [])

collections_df = pd.DataFrame(
    [
        {
            "id": item.get("@id"),
            "title": item.get("title"),
            "type": item.get("@type"),
            "total_items": item.get("totalItems"),
            "description": item.get("description"),
        }
        for item in collections
    ]
)

collections_df

,id,title,type,total_items,description
0,ENCPOS,Les positions des thèses de l'Ecole nationale ...,Collection,None,NaN
1,theater,Théâtre classique,Collection,None,Publier une pièce de théâtre. Cette recette dé...
2,sanctoral_elec,Sanctoral,Collection,None,Accordant à la liturgie une place centrale dan...
3,obituaires,Obituaire,Collection,None,"En 1406, Guillaume de la Perche, prieur de la ..."
4,comptes,Les comptes des consuls de Montferrand,Collection,None,Les archives médiévales de Montferrand sont pa...
5,formulaires,Le formulaire d'Odart Morchesne,Collection,None,"Achevé au plus tard au début de 1427, le formu..."
6,dubourg,Correspondance d'Antoine Du Bourg,Collection,None,Antoine Du Bourg (v. 1490-1538) a fait une car...
7,miroir,Le Miroir des classiques,Collection,None,Le Miroir des classiques est un répertoire des...
8,cjc,Corpus Juris Civilis,Collection,None,Corpus juris civilis regroupe des compilations...
9,edits,Édits de pacification,Collection,None,Édition électronique des édits de pacification...


The `id` column contains the identifiers that can be passed back to the Collection endpoint or to a DTS client.

For this example, we will retrieve the collection whose identifier is `cid` corresponding to [Actes des congrès de la commission internationale de diplomatique (1972-2010)](http://corpus.enc.sorbonne.fr/cid/) corpora.

Before starting the complete download, we can verify that the collection is advertised by the service.

In [6]:
collection_id = "cid"

matching_collections = collections_df.loc[
    collections_df["id"].eq(collection_id)
]

if matching_collections.empty:
    available_ids = (
        collections_df["id"]
        .dropna()
        .astype(str)
        .tolist()
    )

    raise ValueError(
        f"Collection {collection_id!r} was not found. "
        f"Available collection IDs: {available_ids}"
    )

matching_collections

,id,title,type,total_items,description
12,cid,Actes des congrès de la Commission internation...,Collection,None,"Fondée de façon informelle en 1965, la Commiss..."


#### Retrieving the collection with ThunderDots

We now create a `ThunderDots` client.

The `endpoint_dts` argument identifies the DTS service. The `collection_params` dictionary tells ThunderDots which collection to traverse.

Calling `fetch()` makes ThunderDots:

1. explore the selected collection;
2. discover its textual resources;
3. retrieve their documents;
4. extract their text and metadata;
5. store the resulting records in the client.

In [7]:
td = ThunderDots(
    endpoint_dts=DTS_ENDPOINT,
    # In context of Jupyter notebook we set verbose=False to deactivate tool UI
    verbose=False,
    collection_params={
        "collection_id": "cid",
    },
    resource_params={
        # Retrieve one complete textual record per DTS resource. (check other fragmentation mode: https://dots-suite.github.io/ThunderDots/fragmentation/)
        "fragment_mode": "document",
        "add_head_to_content": False
    },
)

td.fetch()

In [8]:
# Optional — inspect raw results without printing the full object.
results = td.results()

print(type(results))
print(
    "Resources:",
    len(results.get("resource_results", [])),
)

<class 'dict'>
Resources: 14


#### Exporting the collection to a Pandas DataFrame

ThunderDots can flatten all fetched resources into a tabular representation.

The method:

```python
df = td.to_dataframe(backend="pandas")
```

creates one row per DTS resource.

By default, the exported table can contain:

- `id`: DTS resource identifier;
- `type`: resource type;
- `title`: resource title;
- `linked_parents`: parent collection identifiers;
- `fragments_count`: number of extracted text fragments;
- `text`: complete text assembled from the fragments;
- flattened Dublin Core metadata such as `dublincore.creator`;
- flattened extension metadata.

See the [ThunderDots DataFrame export documentation](https://dots-suite.github.io/ThunderDots/exports/#pandas-polars-dataframe).

In [9]:
df = td.to_dataframe(
    backend="pandas",
)

print(df.shape)
df.head()

(14, 54)


,id,type,title,linked_parents,fragments_count,text,dublincore.contributor,dublincore.created,dublincore.creator,dublincore.description,...,extensions.funder,extensions.inLanguage,extensions.license,extensions.name,extensions.@type,extensions.editor.@type,extensions.editor.@id,extensions.editor.name,extensions.@context.sameAs,extensions.editor.sameAs
0,cid1994,Resource,Estudios sobre el Notariado Europeo (siglos XI...,[cid],1,"Pilar Ostos, Universidad de Sevilla María Luis...","[Ostos, Pilar, Pardo Rodríguez, María Luisa]",1994,Commission internationale de diplomatique,Ce fichier est issu d'une saisie par la sociét...,...,École nationale des chartes,"[es, fr, pt, en]",https://creativecommons.org/licenses/by-nc-sa/...,Estudios sobre el Notariado Europeo (siglos XI...,Book,NaN,NaN,NaN,NaN,NaN
1,cid1992,Resource,Typologie der Königsurkunden [VIIe-milieu XIII...,[cid],1,Hartmut Atsma et Jean Vezin En raison de notre...,"Bistřický, Jan",1992,Commission internationale de diplomatique,Ce fichier est issu d'une saisie par la sociét...,...,École nationale des chartes,"[fr, es, de, en]",https://creativecommons.org/licenses/by-nc-sa/...,Typologie der Königsurkunden [VIIe-milieu XIII...,Book,schema:Person,https://www.wikidata.org/wiki/Q1681625,"Bistřický, Jan",NaN,NaN
2,cid2009,Resource,Regionale Urkundenbücher,[cid],1,Willibald Rosner Der hier vorgelegte Band 14 d...,"[Kölzer, Theo, Rosner, Willibald, Zehetmayer, ...",2009,Commission internationale de diplomatique,NaN,...,École nationale des chartes,"[de, it, fr, en, es]",https://creativecommons.org/licenses/by-nc-sa/...,Regionale Urkundenbücher,Book,NaN,NaN,NaN,NaN,NaN
3,cid1985,Resource,Cancelleria e cultura nel Medio Evo,[cid],1,Robert-Henri Bautier Le rapport qui doit servi...,"Gualdo, Germano",1985,"[Congresso internazionale di scienze storiche,...",NaN,...,École nationale des chartes,"[it, fr, de, es, en]",https://creativecommons.org/licenses/by-nc-sa/...,Cancelleria e cultura nel Medio Evo,Book,schema:Person,NaN,"Gualdo, Germano",NaN,NaN
4,cid1991,Resource,Diplomatique royale du Moyen Âge XIIIe-XIVe si...,[cid],1,Robert-Henri Bautier Avant de dresser une typo...,"Marques, José",1991,Commission internationale de diplomatique,Ce fichier est issu d'une saisie par la sociét...,...,École nationale des chartes,"[fr, es, de, en]",https://creativecommons.org/licenses/by-nc-sa/...,Diplomatique royale du Moyen Âge XIIIe-XIVe si...,Book,schema:Person,NaN,"Marques, José",NaN,NaN


Check all columns in `list`, by default thunderdots retrieve all metadata.

In [10]:
df.columns.tolist()

['id',
 'type',
 'title',
 'linked_parents',
 'fragments_count',
 'text',
 'dublincore.contributor',
 'dublincore.created',
 'dublincore.creator',
 'dublincore.description',
 'dublincore.identifier',
 'dublincore.issued',
 'dublincore.language',
 'dublincore.license',
 'dublincore.publisher',
 'dublincore.rights',
 'dublincore.source',
 'dublincore.title',
 'extensions.@context.dots',
 'extensions.@context.schema',
 'extensions.@context.contentUrl',
 'extensions.@context.dateCreated',
 'extensions.@context.datePublished',
 'extensions.@context.description',
 'extensions.@context.editor',
 'extensions.@context.encoding',
 'extensions.@context.encodingFormat',
 'extensions.@context.funder',
 'extensions.@context.inLanguage',
 'extensions.@context.isBasedOn',
 'extensions.@context.license',
 'extensions.@context.name',
 'extensions.@context.publisher',
 'extensions.editor',
 'extensions.isBasedOn.@type',
 'extensions.isBasedOn.@id',
 'extensions.isBasedOn.name',
 'extensions.publisher.@ty

We want to keep these metadata only

In [11]:
preferred_columns = [
    "id",
    "title",
    "dublincore.creator",
    "dublincore.contributor",
    "extensions.inLanguage",
    "dublincore.created", 
    "fragments_count",
    "text",
]

available_columns = [
    column
    for column in preferred_columns
    if column in df.columns
]

df[available_columns].head()

,id,title,dublincore.creator,dublincore.contributor,extensions.inLanguage,dublincore.created,fragments_count,text
0,cid1994,Estudios sobre el Notariado Europeo (siglos XI...,Commission internationale de diplomatique,"[Ostos, Pilar, Pardo Rodríguez, María Luisa]","[es, fr, pt, en]",1994,1,"Pilar Ostos, Universidad de Sevilla María Luis..."
1,cid1992,Typologie der Königsurkunden [VIIe-milieu XIII...,Commission internationale de diplomatique,"Bistřický, Jan","[fr, es, de, en]",1992,1,Hartmut Atsma et Jean Vezin En raison de notre...
2,cid2009,Regionale Urkundenbücher,Commission internationale de diplomatique,"[Kölzer, Theo, Rosner, Willibald, Zehetmayer, ...","[de, it, fr, en, es]",2009,1,Willibald Rosner Der hier vorgelegte Band 14 d...
3,cid1985,Cancelleria e cultura nel Medio Evo,"[Congresso internazionale di scienze storiche,...","Gualdo, Germano","[it, fr, de, es, en]",1985,1,Robert-Henri Bautier Le rapport qui doit servi...
4,cid1991,Diplomatique royale du Moyen Âge XIIIe-XIVe si...,Commission internationale de diplomatique,"Marques, José","[fr, es, de, en]",1991,1,Robert-Henri Bautier Avant de dresser une typo...


We want to rename our preffered columns with human-readable labels

In [12]:
# Optional — select and rename columns during export.
column_map = {
    "id": "resource_id",
    "title": "title",
    "dublincore.creator": "creator",
    "dublincore.contributor":"contributor",
    "extensions.inLanguage": "language",
    "dublincore.created": "date",
    "text": "full_text",
}

# Keep only mappings whose source column exists in this collection.
available_column_map = {
    source: target
    for source, target in column_map.items()
    if source in df.columns
}

cid_df = td.to_dataframe(
    backend="pandas",
    column_map=available_column_map,
)

cid_df.head()

,resource_id,title,creator,contributor,language,date,full_text
0,cid1994,Estudios sobre el Notariado Europeo (siglos XI...,Commission internationale de diplomatique,"[Ostos, Pilar, Pardo Rodríguez, María Luisa]","[es, fr, pt, en]",1994,"Pilar Ostos, Universidad de Sevilla María Luis..."
1,cid1992,Typologie der Königsurkunden [VIIe-milieu XIII...,Commission internationale de diplomatique,"Bistřický, Jan","[fr, es, de, en]",1992,Hartmut Atsma et Jean Vezin En raison de notre...
2,cid2009,Regionale Urkundenbücher,Commission internationale de diplomatique,"[Kölzer, Theo, Rosner, Willibald, Zehetmayer, ...","[de, it, fr, en, es]",2009,Willibald Rosner Der hier vorgelegte Band 14 d...
3,cid1985,Cancelleria e cultura nel Medio Evo,"[Congresso internazionale di scienze storiche,...","Gualdo, Germano","[it, fr, de, es, en]",1985,Robert-Henri Bautier Le rapport qui doit servi...
4,cid1991,Diplomatique royale du Moyen Âge XIIIe-XIVe si...,Commission internationale de diplomatique,"Marques, José","[fr, es, de, en]",1991,Robert-Henri Bautier Avant de dresser une typo...


We restrict the corpus to resources published **after 1990** and containing
French (`fr`) among their declared languages.

The `language(s)` column may contain either a list of language codes or a
string representation of such a list, so the filtering function handles both
formats. We then report the number of retained resources and the number of
resources containing non-empty full text.

In [13]:
import ast

def contains_french(value: object) -> bool:
    """Return True when a language field contains the `fr` code."""
    if isinstance(value, (list, tuple, set)):
        return "fr" in value

    if isinstance(value, str):
        value = value.strip()

        # Handle values serialized as strings, such as "['fr', 'en']".
        if value.startswith("["):
            try:
                parsed_value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed_value = None

            if isinstance(parsed_value, (list, tuple, set)):
                return "fr" in parsed_value

        # Handle simpler strings such as "fr", "fr, en", or "fr es de".
        language_codes = {
            language.strip().lower()
            for language in value.replace(",", " ").split()
        }
        return "fr" in language_codes

    return False

filtered_df = cid_df

# Convert dates to numeric values; invalid or missing dates become NaN.
filtered_df["date"] = pd.to_numeric(
    filtered_df["date"],
    errors="coerce",
)

# Keep resources published after 1990 and containing French.
filtered_df = (
    filtered_df.loc[
        filtered_df["date"].gt(1990) # gt = greater than
        & filtered_df["language"].apply(contains_french)
    ]
    .copy()
    .reset_index(drop=True)
)

# Basic corpus checks.
print(f"Initial number of resources: {len(cid_df)}")
print(f"Number of filtered resources: {len(filtered_df)}")

if "full_text" in filtered_df.columns:
    non_empty_texts = (
        filtered_df["full_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        "Filtered resources with non-empty text:",
        non_empty_texts,
    )

filtered_df.head()

Initial number of resources: 14
Number of filtered resources: 10
Filtered resources with non-empty text: 10


,resource_id,title,creator,contributor,language,date,full_text
0,cid1994,Estudios sobre el Notariado Europeo (siglos XI...,Commission internationale de diplomatique,"[Ostos, Pilar, Pardo Rodríguez, María Luisa]","[es, fr, pt, en]",1994,"Pilar Ostos, Universidad de Sevilla María Luis..."
1,cid1992,Typologie der Königsurkunden [VIIe-milieu XIII...,Commission internationale de diplomatique,"Bistřický, Jan","[fr, es, de, en]",1992,Hartmut Atsma et Jean Vezin En raison de notre...
2,cid2009,Regionale Urkundenbücher,Commission internationale de diplomatique,"[Kölzer, Theo, Rosner, Willibald, Zehetmayer, ...","[de, it, fr, en, es]",2009,Willibald Rosner Der hier vorgelegte Band 14 d...
3,cid1991,Diplomatique royale du Moyen Âge XIIIe-XIVe si...,Commission internationale de diplomatique,"Marques, José","[fr, es, de, en]",1991,Robert-Henri Bautier Avant de dresser une typo...
4,cid1996,Papsturkunde und europäisches Urkundenwesen,Commission internationale de diplomatique,"[Herde, Peter, Jakobs, Hermann, Olivier Guyotj...","[de, it, fr, en, es]",1996,Rudolf Hiestand Als Hintergrund für die Einflü...


Inspect the full text of random resource

In [14]:
row = cid_df.sample(n=1).iloc[0]
contributors = row["contributor"]
authors = (
    contributors
    if isinstance(contributors, str)
    else "; ".join(map(str, contributors))
)
print(
    f"ID: {row['resource_id']} | "
    f"Title: {row['title']} | "
    f"Date: {row['date']} | "
    f"Authors: {authors}\n"
    f"Extract:\n{str(row['full_text'])[:500]}..."
)

ID: cid1986 | Title: Notariado público y documento privado: de los orígenes al siglo XIV | Date: 1986 | Authors: Trenchs, José; Olivier Guyotjeannin
Extract:
Angel Canellas Lopez El estudio del desarrollo y difusión del notariado público, y del documento privado, desde los orígenes hasta el siglo xiv, tema central del VII Congreso Internacional de Diplomática, aplicado al área geofráfica de la Península Ibérica, exige plantear el examen del panorama de la investigación sobre la génesis y evolución de un incipiente notariado, sobre la documentación engendrada y conservada durante los primeros siglos de nuestra Edad Media, trazar el significado de las ...


In [15]:
# Optional — save to disk.
"""
df.to_csv(
    "../output/cid_dts_collection.tsv",
    sep="\t",
    encoding="utf-8",
    index=False,
)
"""

'\ndf.to_csv(\n    "../output/cid_dts_collection.tsv",\n    sep="\t",\n    encoding="utf-8",\n    index=False,\n)\n'

#### Conclusion

DTS gives us a standardized way to discover and retrieve textual collections, while ThunderDots handles much of the repetitive work required to traverse the service, download documents and normalize their metadata.

The complete workflow is:

```text
DTS entry point
        ↓
Collection endpoint
        ↓
collection identifier (`cid`)
        ↓
ThunderDots traversal and document retrieval
        ↓
Pandas DataFrame
```

This is particularly useful for building research corpora because the same client can be reused with other DTS-compatible repositories.